In [2]:
import pandas as pd
import numpy as np
import matplotlib as plt
from functions import clean_data, split_data

In [3]:
X, Y = clean_data("claims_train.csv")
X_train, X_val, y_train, y_val = split_data(X, Y, 0.2)

In [ ]:

# Checking shapes just in case
# print(X_train.shape)
# print(y_train.shape)

(433928, 41)
(433928,)


In [ ]:
# Initialize weights(random) and biases(0)
# Using He initialization for the weights

def initialize_parameters(input_layer, hidden_layer1_neurons, hidden_layer2_neurons, output_layer):
    np.random.seed(42)
    parameters = {

    "w1": np.random.randn(input_layer, hidden_layer1_neurons) * np.sqrt(2.0 / input_layer),
    "b1": np.zeros((1, hidden_layer1_neurons)),

    "w2":np.random.randn(hidden_layer1_neurons, hidden_layer2_neurons) * np.sqrt(2.0 / hidden_layer1_neurons),
    "b2": np.zeros((1, hidden_layer2_neurons)), 

    "w3": np.random.randn(hidden_layer2_neurons,output_layer) * np.sqrt(2.0 / hidden_layer2_neurons),
    "b3": np.zeros((1, output_layer))}
    return parameters

In [ ]:
# Activation function (in our case - ReLU and softplus for output layer)
# It keeps positive values and turns negative values into 0

def relu(z):
    return np.maximum(0, z)

def softplus(z):
    return np.log1p(np.exp(z))

def relu_derivative(z):
    return (z > 0).astype(float)

def softplus_derivative(z):
    # d/dz softplus(z) = sigmoid(z)
    return 1 / (1 + np.exp(-z))

In [2]:
batch_sizes = [128, 256, 512]
epoch_options = [50, 70, 100]

In [ ]:
def feed_forward(X, parameters):
    w1, b1 = parameters["w1"], parameters["b1"]
    w2, b2 = parameters["w2"], parameters["b2"]
    w3, b3 = parameters["w3"], parameters["b3"]
    
    # first layer
    z1 = X @ w1 + b1
    a1 = relu(z1)
    
    # second layer
    z2 = a1 @ w2 + b2
    a2 = relu(z2)
    
    # third layer
    z3 = a2 @ w3 + b3
    predicted_y = softplus(z3)
    
    values = {"z1": z1, "a1": a1, "z2": z2, "a2": a2, "z3": z3, "predicted_y": predicted_y}
    return predicted_y, values
    

In [ ]:
def backward_propagation(X, y, parameters, values, huber = True, delta = 1.0):
    w1, b1 = parameters["w1"], parameters["b1"]
    w2, b2 = parameters["w2"], parameters["b2"]
    w3, b3 = parameters["w3"], parameters["b3"]
    
    z1, a1 = values["z1"], values["a1"]
    z2, a2 = values["z2"], values["a2"]
    z3 = values["z3"]
    
    y_pred = values["predicted_y"]
    m = X.shape[0]
    
    # 1. error
    error = y - y_pred
    absolute_err = np.abs(error)

    # 2. huber gradient dL/d(y_pred)
    dL_dy = np.where(
        absolute_err <= delta,
        -error,                     # small errors -> quadratic
        -delta * np.sign(error)     # big errors -> linear
    )

    # y_pred = softplus(z3)
    dz3 = dL_dy * softplus_derivative(z3)         # (m, 1)
    dw3 = (a2.T @ dz3) / m                        # (hidden2, 1)
    db3 = np.sum(dz3, axis=0, keepdims=True) / m  # (1, 1)

    #  3. Hidden layer 2 (ReLU) 
    da2 = dz3 @ w3.T                              # (m, hidden2)
    dz2 = da2 * relu_derivative(z2)               # (m, hidden2)
    dw2 = (a1.T @ dz2) / m                        # (hidden1, hidden2)
    db2 = np.sum(dz2, axis=0, keepdims=True) / m  # (1, hidden2)

    # 4. Hidden layer 1 (ReLU) 
    da1 = dz2 @ w2.T                              # (m, hidden1)
    dz1 = da1 * relu_derivative(z1)               # (m, hidden1)
    dw1 = (X.T @ dz1) / m                         # (input, hidden1)
    db1 = np.sum(dz1, axis=0, keepdims=True) / m  # (1, hidden1)

    grads = {
        "w1": dw1,
        "b1": db1,
        "w2": dw2,
        "b2": db2,
        "w3": dw3,
        "b3": db3,
    }
    return grads

In [ ]:
def init_adam_state(parameters):
    adam_state = {
        "t": 0,
        "m": {},
        "v": {}
    }
    for name, value in parameters.items():
        adam_state["m"][name] = np.zeros_like(value)
        adam_state["v"][name] = np.zeros_like(value)
    return adam_state


In [ ]:
def adam_update(parameters, grads, adam_state,
                learning_rate=0.001,
                beta1=0.9, beta2=0.999, eps=1e-8):
    adam_state["t"] += 1
    t = adam_state["t"]

    for name in parameters.keys():
        g = grads[name]

        # first moment
        adam_state["m"][name] = beta1 * adam_state["m"][name] + (1 - beta1) * g
        # second moment
        adam_state["v"][name] = beta2 * adam_state["v"][name] + (1 - beta2) * (g ** 2)

        m_hat = adam_state["m"][name] / (1 - beta1 ** t)
        v_hat = adam_state["v"][name] / (1 - beta2 ** t)

        parameters[name] -= learning_rate * m_hat / (np.sqrt(v_hat) + eps)

    return parameters, adam_state


In [ ]:
def train_model(X_train, y_train, X_val, y_val,
                learning_rate, batch_size, epochs, delta=1.0):

    # 1. initialize parameters
    parameters = initialize_parameters(
        input_layer = X_train.shape[1],
        hidden_layer1_neurons = 28,
        hidden_layer2_neurons = 28,
        output_layer = 1
    )
    adam_state = init_adam_state(parameters)
    best_val_loss = float('inf')
    patience_counter = 0
    best_parameters = {k: v.copy() for k, v in parameters.items()}

    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        # shuffle at start of epoch
        indices = np.arange(len(X_train))
        np.random.shuffle(indices)
        X_train = X_train[indices]
        y_train = y_train[indices]
        
        batch_losses = []
        
        # Training in mini-batches
        
        for i in range(0, len(X_train), batch_size):

            X_batch = X_train[i : i + batch_size]
            y_batch = y_train[i : i + batch_size]

            # forward
            predicted_y, values = feed_forward(X_batch, parameters)
            
            # compute TRAINING error and loss (IMPORTANT)
            train_error = y_batch - predicted_y
            train_loss = np.mean(np.where(
                np.abs(train_error) <= delta,
                0.5 * train_error**2,
                delta * (np.abs(train_error) - 0.5 * delta)
            )) 
            batch_losses.append(train_loss)

            grads = backward_propagation(
        X_batch, y_batch, parameters, values,
        huber=True, delta=delta
        )

        # 4. Adam update – THIS is where parameters change now
        parameters, adam_state = adam_update(
        parameters, grads, adam_state,
        learning_rate=learning_rate  
        )
            
        epoch_train_loss = np.mean(batch_losses)
        train_losses.append(epoch_train_loss)
        
        # Validation loss
        
        y_val_pred, _ = feed_forward(X_val, parameters)
        val_error = y_val - y_val_pred
        val_loss = np.mean( np.where(
            np.abs(val_error) <= delta,
            0.5 * val_error**2,
            delta * (np.abs(val_error) - 0.5 * delta)
        ))
        val_losses.append(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_parameters = {k: v.copy() for k, v in parameters.items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= 5:   # or your patience variable
            break
        
        print(f"Epoch {epoch}: train={epoch_train_loss:.4f}, val={val_loss:.4f}")
        
    return best_parameters, train_losses, val_losses


In [ ]:
best_config = None
best_loss = float("inf")

for batch_size in batch_sizes:
    for epochs in epoch_options:

        print(f"Testing batch={batch_size}, epochs={epochs}")

        val_loss = train_model(
            X_train, y_train,
            X_val, y_val,
            learning_rate=0.001,
            batch_size=batch_size,
            epochs=epochs,
            delta=1.0
        )

        print("Validation loss:", val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            best_config = (batch_size, epochs)

print("\nBest hyperparameters:")
print(best_config)
print("Best validation loss:", best_loss)
